In [172]:
import pandas as pd
import numpy as np
import re,yaml,os
import itertools as it
import networkx as nx
def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [173]:
pd.__version__
%store -r num
# num=0
print(num)

4


In [174]:
rep="/Volumes/BroadExt/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
rep="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
fParadigmes="vlexique2-CV5-Test%d.csv"%num
paradigmes=pd.read_csv(rep+fParadigmes,sep=";",encoding="utf8")

In [175]:
cols=paradigmes.columns.tolist()
cases=cols[:]
cases.remove("lexeme")
print(len(cases),", ".join(cases))

51 ai1P, ai1S, ai2P, ai2S, ai3P, ai3S, fi1P, fi1S, fi2P, fi2S, fi3P, fi3S, ii1P, ii1S, ii2P, ii2S, ii3P, ii3S, inf, is1P, is1S, is2P, is2S, is3P, is3S, pI1P, pI2P, pI2S, pP, pc1P, pc1S, pc2P, pc2S, pc3P, pc3S, pi1P, pi1S, pi2P, pi2S, pi3P, pi3S, ppFP, ppFS, ppMP, ppMS, ps1P, ps1S, ps2P, ps2S, ps3P, ps3S


In [176]:
for c in cases:
    display(paradigmes[[c]].dropna().head(1))

,ai1P
100,afirmam


,ai1S
13,abuti


,ai2P
0,abEsat


,ai2S
51,ak2ji


,ai3P
45,akuryr


...

,ps1S
3,abat


,ps2P
13,abutisje


,ps2S
4,abdik


,ps3P
3,abat


,ps3S
8,abôd


In [177]:
g=nx.Graph()
for (c1,c2) in it.combinations(cases, 2):
    c1Val=paradigmes[c1].notnull()
    c2Val=paradigmes[c2].notnull()
    incompatibles=paradigmes[c1Val & c2Val & (paradigmes[c1]!=paradigmes[c2])][[c1,c2]]
    compatibles=paradigmes[c1Val & c2Val & (paradigmes[c1]==paradigmes[c2])][[c1,c2]]
    if len(incompatibles)==0 and len(compatibles)>0:
        g.add_edge(c1,c2,weight=len(compatibles))
cliques=list(nx.find_cliques(g))

In [178]:
cliques=sorted(cliques, key=len, reverse=True)
print(cliques)
maxLenClique=len(cliques[0])
wCliques=(cliques[:])

[['pc3P', 'pc2S', 'fi1S', 'pc3S'], ['pc3P', 'pc2S', 'fi1S', 'pc1S'], ['ii3P', 'ii1S', 'ii2S', 'ii3S'], ['is1S', 'is2S', 'ps2S', 'ps3S'], ['is1S', 'ps3P', 'ps2S', 'ps3S'], ['ps1S', 'ps2S', 'ps3P', 'ps3S'], ['is2P', 'ps2P', 'ii2P'], ['ps1P', 'is1P', 'ii1P'], ['is1S', 'is2S', 'is3P'], ['pi2S', 'pI2S'], ['pi3S', 'pI2S'], ['is3S', 'ai3S'], ['ppFS', 'ppFP'], ['ppMS', 'ppMP'], ['pI1P', 'pi1P'], ['ai3S', 'ai2S'], ['is1S', 'pi3P'], ['ps1S', 'pi3P'], ['ps1S', 'is3P']]


In [179]:
syncretismes=[]

def cleanCliques(lCliques):
    sCases=set(sum(lCliques,[]))
    for c in lCliques:
        wCliques.remove(c)
    for c in sCases:
        for w in wCliques:
            if c in w:
                w.remove(c)
    return
                

def addCliques(length):
    conflits=[]
    lCliques=[x for x in cliques if len(x)==length]
    if len(lCliques)>1:
        for (c1,c2) in it.combinations(lCliques, 2):
            inter=set(c1).intersection(set(c2))
            if inter:
                conflits.append([c1,c2])
    if conflits:
        ajouts=[]
        for c in lCliques:
            noConflit=True
            for conflit in conflits:
                if c in conflit:
                    noConflit=False
            if noConflit:
                ajouts.append(c)
        syncretismes.extend(ajouts)
        cleanCliques(ajouts)
        for c1,c2 in conflits:
            sC1=set(c1)
            sC2=set(c2)
            pivot=list(sC1.intersection(sC2))[0]
            dC1=sC1.difference(sC2)
            wC1=sum(g[pivot][c]["weight"] for c in dC1)
            dC2=sC2.difference(sC1)
            wC2=sum(g[pivot][c]["weight"] for c in dC2)
            print (c1,wC1,c2,wC2)
    else:
        syncretismes.extend(lCliques)
        cleanCliques(lCliques)
    return

In [180]:
for i in range(maxLenClique):
    if maxLenClique-i>1:
        print(maxLenClique-i)
        addCliques(maxLenClique-i)

print(syncretismes)
syncretiques=set()
for l in syncretismes:
    print(l)
    for c in l:
        print(c)
        syncretiques.add(c)

syncretismes,syncretiques,wCliques

4
['pc3P', 'pc2S', 'fi1S', 'pc3S'] 64 ['pc3P', 'pc2S', 'fi1S', 'pc1S'] 60
['is1S', 'is2S', 'ps2S', 'ps3S'] 1 ['is1S', 'ps3P', 'ps2S', 'ps3S'] 3
['is1S', 'is2S', 'ps2S', 'ps3S'] 4 ['ps1S', 'ps2S', 'ps3P', 'ps3S'] 154
['is1S', 'ps3P', 'ps2S', 'ps3S'] 1 ['ps1S', 'ps2S', 'ps3P', 'ps3S'] 88
3
2
['pi2S', 'pI2S'] 126 ['pi3S', 'pI2S'] 146
['is3S', 'ai3S'] 6 ['ai3S', 'ai2S'] 8
[['ii3P', 'ii1S', 'ii2S', 'ii3S'], ['is2P', 'ps2P', 'ii2P'], ['ps1P', 'is1P', 'ii1P'], ['is1S', 'is2S', 'is3P'], ['ps2S', 'ps3S'], ['ppFS', 'ppFP'], ['ppMS', 'ppMP'], ['pI1P', 'pi1P'], ['ps1S', 'pi3P']]
['ii3P', 'ii1S', 'ii2S', 'ii3S']
ii3P
ii1S
ii2S
ii3S
['is2P', 'ps2P', 'ii2P']
is2P
ps2P
ii2P
['ps1P', 'is1P', 'ii1P']
ps1P
is1P
ii1P
['is1S', 'is2S', 'is3P']
is1S
is2S
is3P
['ps2S', 'ps3S']
ps2S
ps3S
['ppFS', 'ppFP']
ppFS
ppFP
['ppMS', 'ppMP']
ppMS
ppMP
['pI1P', 'pi1P']
pI1P
pi1P
['ps1S', 'pi3P']
ps1S
pi3P


([['ii3P', 'ii1S', 'ii2S', 'ii3S'],
  ['is2P', 'ps2P', 'ii2P'],
  ['ps1P', 'is1P', 'ii1P'],
  ['is1S', 'is2S', 'is3P'],
  ['ps2S', 'ps3S'],
  ['ppFS', 'ppFP'],
  ['ppMS', 'ppMP'],
  ['pI1P', 'pi1P'],
  ['ps1S', 'pi3P']],
 {'ii1P',
  'ii1S',
  'ii2P',
  'ii2S',
  'ii3P',
  'ii3S',
  'is1P',
  'is1S',
  'is2P',
  'is2S',
  'is3P',
  'pI1P',
  'pi1P',
  'pi3P',
  'ppFP',
  'ppFS',
  'ppMP',
  'ppMS',
  'ps1P',
  'ps1S',
  'ps2P',
  'ps2S',
  'ps3S'},
 [['pc3P', 'pc2S', 'fi1S', 'pc3S'],
  ['pc3P', 'pc2S', 'fi1S', 'pc1S'],
  ['ps3P'],
  ['ps3P'],
  ['pi2S', 'pI2S'],
  ['pi3S', 'pI2S'],
  ['is3S', 'ai3S'],
  ['ai3S', 'ai2S'],
  [],
  []])

In [181]:
dExpansion={s[0]:s for s in syncretismes if len(s)>1}
dExpansion

{'ii3P': ['ii3P', 'ii1S', 'ii2S', 'ii3S'],
 'is2P': ['is2P', 'ps2P', 'ii2P'],
 'ps1P': ['ps1P', 'is1P', 'ii1P'],
 'is1S': ['is1S', 'is2S', 'is3P'],
 'ps2S': ['ps2S', 'ps3S'],
 'ppFS': ['ppFS', 'ppFP'],
 'ppMS': ['ppMS', 'ppMP'],
 'pI1P': ['pI1P', 'pi1P'],
 'ps1S': ['ps1S', 'pi3P']}

In [182]:
if "fi2S" in dExpansion:
    if "fi3S" in dExpansion["fi2S"]:
        dExpansion[u"fi3S"]=dExpansion.pop("fi2S")
if "ii1S" in dExpansion:
    if "ii3S" in dExpansion["ii1S"]:
        dExpansion[u"ii3S"]=dExpansion.pop("ii1S")
if "pc2S" in dExpansion:
    if "pc3S" in dExpansion["pc2S"]:
        dExpansion[u"pc3S"]=dExpansion.pop("pc2S")
if "ps2S" in dExpansion:
    if "ps3S" in dExpansion["ps2S"]:
        dExpansion[u"ps3S"]=dExpansion.pop("ps2S")        
if "pi2S" in dExpansion:
    if "pi3S" in dExpansion["pi2S"]:
        dExpansion[u"pi3S"]=dExpansion.pop("pi2S")
if "ppMP" in dExpansion:
    if "ppMS" in dExpansion["ppMP"]:
        dExpansion[u"ppMS"]=dExpansion.pop("ppMP")

In [183]:
dExpansion

{'ii3P': ['ii3P', 'ii1S', 'ii2S', 'ii3S'],
 'is2P': ['is2P', 'ps2P', 'ii2P'],
 'ps1P': ['ps1P', 'is1P', 'ii1P'],
 'is1S': ['is1S', 'is2S', 'is3P'],
 'ppFS': ['ppFS', 'ppFP'],
 'ppMS': ['ppMS', 'ppMP'],
 'pI1P': ['pI1P', 'pi1P'],
 'ps1S': ['ps1S', 'pi3P'],
 'ps3S': ['ps2S', 'ps3S']}

In [184]:
print([c for c in cols if c not in syncretiques])
omps=paradigmes[[c for c in cols if c not in syncretiques]].copy()

['lexeme', 'ai1P', 'ai1S', 'ai2P', 'ai2S', 'ai3P', 'ai3S', 'fi1P', 'fi1S', 'fi2P', 'fi2S', 'fi3P', 'fi3S', 'inf', 'is3S', 'pI2P', 'pI2S', 'pP', 'pc1P', 'pc1S', 'pc2P', 'pc2S', 'pc3P', 'pc3S', 'pi1S', 'pi2P', 'pi2S', 'pi3S', 'ps3P']


In [185]:
def fusionFormes(row):
    result=np.nan
    for c in row:
        if c==c:
            result=c
            break
    return result

In [186]:
# paradigmes[["ppMS","ppMP"]].apply(fusionFormes,axis=1)

In [187]:
for s in dExpansion:
    print (s,dExpansion[s])
    omps[s]=paradigmes[dExpansion[s]].apply(fusionFormes,axis=1)

ii3P ['ii3P', 'ii1S', 'ii2S', 'ii3S']
is2P ['is2P', 'ps2P', 'ii2P']
ps1P ['ps1P', 'is1P', 'ii1P']
is1S ['is1S', 'is2S', 'is3P']
ppFS ['ppFS', 'ppFP']
ppMS ['ppMS', 'ppMP']
pI1P ['pI1P', 'pi1P']
ps1S ['ps1S', 'pi3P']
ps3S ['ps2S', 'ps3S']


In [188]:
with open(rep+fParadigmes.replace(".csv","-omp.yaml"),"w") as outFile:
    yaml.safe_dump(dExpansion,outFile)

In [189]:
omps.dropna(thresh=3).to_csv(rep+fParadigmes.replace(".csv","-omp.csv"),encoding="utf8",sep=";",index=None)

In [190]:
if num<8:
    print(num)
    num+=1
    ding()
else:
    num=0
    ding()
    ding()
    ding()
%store num

4
Stored 'num' (int)
